# Build Classified ChatDev Scored 1 Dataset

Score summarized ChatDev JSON files with a selectable premise subset, then write the results under `data/classified_chatdev_scored_1`. Change `PREMISE_ORDER` to choose which groups from `scoring_context.premise_order` are used by the judge, for example `["tasks"]`, `["user_demand"]`, or `["user_demand", "tasks"]`.

In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import json
import shutil
import sys
import time

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_ROOT = PROJECT_ROOT / "data" / "classified_chatdev_summarized"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "classified_chatdev_scored_1"
DATASET_JSON = PROJECT_ROOT / "chatdev_dataset.json"
TRAJECTORY_DIR = PROJECT_ROOT / "data" / "classified_chatdev_playbook" / "trajectory"

# Choose any non-empty subset of: user_demand, known_context, tasks.
# For error 1.1 focused on phase/task compliance, start with ["tasks"].
PREMISE_ORDER = ["tasks", "user_demand"]

MODEL = "gpt-4o-mini"
TOP_LOGPROBS = 5
OVERWRITE = False
MAX_WORKERS = 10

# scored_1 only evaluates the 0.0 vs 1.1 task. A JSON is included when any
# same-filename copy appears under one of these labels; unrelated filenames are skipped
# before API scoring to reduce cost.
TARGET_LABELS = {"0.0", "1.1"}

print("Project root:", PROJECT_ROOT)
print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Premise order:", PREMISE_ORDER)
print("Target labels:", sorted(TARGET_LABELS) if TARGET_LABELS is not None else None)
print("Max workers:", MAX_WORKERS)


Project root: d:\Works\code\winter-like-ai\ChatDev
Input root: d:\Works\code\winter-like-ai\ChatDev\data\classified_chatdev_summarized
Output root: d:\Works\code\winter-like-ai\ChatDev\data\classified_chatdev_scored_1
Premise order: ['tasks', 'user_demand']
Target labels: ['0.0', '1.1']
Max workers: 10


In [2]:
from chatdev.analyzer.logprob_consistency import (
    LogprobConsistencyScorer,
    chatdev_filename_sort_key,
    load_user_task_map,
    normalize_premise_order,
)

PREMISE_ORDER = list(normalize_premise_order(PREMISE_ORDER))
TARGET_LABELS = set(TARGET_LABELS) if TARGET_LABELS is not None else None


def scored_output_path(src_path: Path) -> Path:
    rel = src_path.relative_to(INPUT_ROOT)
    name = rel.name
    if name.endswith("_summarized.json"):
        name = name[:-len("_summarized.json")] + "_scored.json"
    else:
        name = rel.stem + "_scored.json"
    return OUTPUT_ROOT / rel.parent / name


def summarized_label(src_path: Path) -> str:
    return src_path.relative_to(INPUT_ROOT).parts[0]


def path_sort_key(path: Path):
    rel = path.relative_to(INPUT_ROOT)
    return (rel.parts[0], chatdev_filename_sort_key(path.name))


def discover_all_summarized_paths():
    paths = [path for path in INPUT_ROOT.rglob("*_summarized.json") if path.is_file()]
    return sorted(paths, key=path_sort_key)


def collect_filename_labels(paths):
    filename_labels = {}
    for path in paths:
        filename_labels.setdefault(path.name, set()).add(summarized_label(path))
    return filename_labels


def discover_summarized_paths():
    paths = discover_all_summarized_paths()
    if TARGET_LABELS is None:
        return paths

    filename_labels = collect_filename_labels(paths)
    return [
        path for path in paths
        if filename_labels.get(path.name, set()) & TARGET_LABELS
    ]


def group_by_filename(paths):
    groups = {}
    for path in paths:
        groups.setdefault(path.name, []).append(path)
    return dict(sorted(groups.items(), key=lambda item: chatdev_filename_sort_key(item[0])))


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def scored_output_has_selected_premises(path: Path) -> bool:
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False
    for interactions in data.values():
        if not isinstance(interactions, list):
            continue
        for entry in interactions:
            if not isinstance(entry, dict):
                continue
            context = entry.get("scoring_context") or {}
            if context.get("premise_order") != PREMISE_ORDER:
                return False
            if "output_consistency_scores" in entry:
                return True
    return False


def copy_scored_json(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


In [3]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
user_task_map = load_user_task_map(dataset_path=str(DATASET_JSON), trajectory_dir=str(TRAJECTORY_DIR))


def score_canonical_file(src_path: Path, dst_path: Path):
    # Use one scorer/client per worker call so concurrent API requests do not share mutable client state.
    worker_scorer = LogprobConsistencyScorer(
        model=MODEL,
        top_logprobs=TOP_LOGPROBS,
        premise_order=PREMISE_ORDER,
    )
    worker_scorer.score_summarized_json(
        input_path=str(src_path),
        output_path=str(dst_path),
        user_task_map=user_task_map,
        dataset_path=str(DATASET_JSON),
        trajectory_dir=str(TRAJECTORY_DIR),
        premise_order=PREMISE_ORDER,
    )
    return worker_scorer.stats

candidate_paths = discover_all_summarized_paths()
candidate_filename_labels = collect_filename_labels(candidate_paths)
all_paths = discover_summarized_paths()
selected_filename_labels = collect_filename_labels(all_paths)
filename_groups = group_by_filename(all_paths)
filename_to_scored = {}

skipped_filenames = sorted(
    set(candidate_filename_labels) - set(selected_filename_labels),
    key=chatdev_filename_sort_key,
)
selected_output_paths = {scored_output_path(path) for path in all_paths}

# Resume support: reuse already scored outputs with the same selected premise order.
for src_path in all_paths:
    dst_path = scored_output_path(src_path)
    if dst_path.exists() and scored_output_has_selected_premises(dst_path):
        filename_to_scored.setdefault(src_path.name, dst_path)

manifest = {
    "input_root": str(INPUT_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "dataset_json": str(DATASET_JSON),
    "trajectory_dir": str(TRAJECTORY_DIR),
    "model": MODEL,
    "top_logprobs": TOP_LOGPROBS,
    "premise_order": PREMISE_ORDER,
    "target_labels": sorted(TARGET_LABELS) if TARGET_LABELS is not None else None,
    "target_label_policy": "Include a filename when any same-filename summarized JSON appears under a target label; skip filenames unrelated to all target labels before scoring.",
    "all_summarized_files": len(candidate_paths),
    "all_unique_filenames": len(candidate_filename_labels),
    "selected_summarized_files": len(all_paths),
    "selected_unique_filenames": len(filename_groups),
    "skipped_unique_filenames": len(skipped_filenames),
    "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "overwrite": OVERWRITE,
    "max_workers": MAX_WORKERS,
    "records": [],
}

print(f"All candidates: {len(candidate_filename_labels)} unique filenames across {len(candidate_paths)} summarized files")
print(f"Selected for 0.0/1.1 task: {len(filename_groups)} unique filenames across {len(all_paths)} summarized files")
print(f"Skipped unrelated filenames before scoring: {len(skipped_filenames)}")
print(f"Already reusable scored filenames: {len(filename_to_scored)}")
print(f"Concurrent scoring workers: {MAX_WORKERS}")

if skipped_filenames[:10]:
    print("Skipped examples:", skipped_filenames[:10])

actions = {}
jobs = []
worker_stats = []

for filename, paths in filename_groups.items():
    canonical_src = paths[0]
    canonical_dst = scored_output_path(canonical_src)
    reusable_path = filename_to_scored.get(filename)

    if reusable_path is not None and not OVERWRITE:
        actions[filename] = "copied_existing"
        if reusable_path != canonical_dst:
            copy_scored_json(reusable_path, canonical_dst)
        filename_to_scored[filename] = canonical_dst
    else:
        jobs.append((filename, canonical_src, canonical_dst))

if jobs:
    print(f"Scoring {len(jobs)} canonical files with {MAX_WORKERS} workers")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_job = {
            executor.submit(score_canonical_file, canonical_src, canonical_dst): (filename, canonical_src, canonical_dst)
            for filename, canonical_src, canonical_dst in jobs
        }
        for completed, future in enumerate(as_completed(future_to_job), start=1):
            filename, canonical_src, canonical_dst = future_to_job[future]
            stats = future.result()
            worker_stats.append(stats)
            actions[filename] = "scored"
            filename_to_scored[filename] = canonical_dst
            print(f"[{completed}/{len(jobs)}] scored {filename}")
else:
    print("No canonical files need API scoring")

for index, (filename, paths) in enumerate(filename_groups.items(), start=1):
    canonical_src = paths[0]
    canonical_dst = scored_output_path(canonical_src)
    action = actions.get(filename, "unknown")

    for duplicate_src in paths[1:]:
        duplicate_dst = scored_output_path(duplicate_src)
        if OVERWRITE or not scored_output_has_selected_premises(duplicate_dst):
            copy_scored_json(canonical_dst, duplicate_dst)

    manifest["records"].append({
        "filename": filename,
        "labels": sorted(selected_filename_labels.get(filename, [])),
        "target_label_hit": sorted((selected_filename_labels.get(filename, set()) & TARGET_LABELS) if TARGET_LABELS is not None else []),
        "canonical_input": str(canonical_src.relative_to(INPUT_ROOT)),
        "canonical_output": str(canonical_dst.relative_to(OUTPUT_ROOT)),
        "copies": len(paths) - 1,
        "action": action,
    })
    if index % 10 == 0 or index == len(filename_groups):
        print(f"[{index}/{len(filename_groups)}] copied duplicates for {filename}: action={action}, copies={len(paths) - 1}")

manifest["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
manifest["scorer_stats"] = {
    "api_calls": sum(stats.get("api_calls", 0) for stats in worker_stats),
    "scored_outputs": sum(stats.get("scored_outputs", 0) for stats in worker_stats),
    "errors": sum(stats.get("errors", 0) for stats in worker_stats),
}
write_json(OUTPUT_ROOT / "_manifest.json", manifest)
print("Done")
print(json.dumps(manifest["scorer_stats"], ensure_ascii=False, indent=2))


All candidates: 130 unique filenames across 448 summarized files
Selected for 0.0/1.1 task: 69 unique filenames across 236 summarized files
Skipped unrelated filenames before scoring: 61
Already reusable scored filenames: 8
Concurrent scoring workers: 10
Skipped examples: ['ChatDev_ProgramDev_GPT4o_3_summarized.json', 'ChatDev_ProgramDev_GPT4o_7_summarized.json', 'ChatDev_ProgramDev_GPT4o_10_summarized.json', 'ChatDev_ProgramDev_GPT4o_12_summarized.json', 'ChatDev_ProgramDev_GPT4o_17_summarized.json', 'ChatDev_ProgramDev_GPT4o_19_summarized.json', 'ChatDev_ProgramDev_GPT4o_21_summarized.json', 'ChatDev_ProgramDev2_GPT4o_3_summarized.json', 'ChatDev_ProgramDev2_GPT4o_7_summarized.json', 'ChatDev_ProgramDev2_GPT4o_10_summarized.json']
Scoring 61 canonical files with 10 workers
[1/61] scored ChatDev_ProgramDev_GPT4o_11_summarized.json
[2/61] scored ChatDev_ProgramDev_GPT4o_24_summarized.json
[3/61] scored ChatDev_ProgramDev_GPT4o_20_summarized.json
[4/61] scored ChatDev_ProgramDev_GPT4o_1

In [4]:
# Quick structural check for the selected 0.0/1.1-related files only.
missing = []
wrong_premises = []
selected_outputs = []
for src_path in discover_summarized_paths():
    dst_path = scored_output_path(src_path)
    selected_outputs.append(dst_path)
    if not dst_path.exists():
        missing.append(str(src_path.relative_to(INPUT_ROOT)))
    elif not scored_output_has_selected_premises(dst_path):
        wrong_premises.append(str(dst_path.relative_to(OUTPUT_ROOT)))

print(f"Selected outputs checked: {len(selected_outputs)}")
print(f"Missing outputs: {len(missing)}")
print(f"Outputs with wrong premise_order: {len(wrong_premises)}")
if missing[:20]:
    print(json.dumps(missing[:20], ensure_ascii=False, indent=2))
if wrong_premises[:20]:
    print(json.dumps(wrong_premises[:20], ensure_ascii=False, indent=2))

sample_outputs = sorted({path for path in selected_outputs if path.exists()})
if sample_outputs:
    sample = sample_outputs[0]
    payload = json.loads(sample.read_text(encoding="utf-8"))
    first_entry = next(
        entry
        for interactions in payload.values()
        if isinstance(interactions, list)
        for entry in interactions
        if isinstance(entry, dict)
    )
    print("Sample:", sample.relative_to(OUTPUT_ROOT))
    print("Sample scoring_context:")
    print(json.dumps(first_entry.get("scoring_context"), ensure_ascii=False, indent=2))


Selected outputs checked: 236
Missing outputs: 0
Outputs with wrong premise_order: 2
[
  "0.0\\ChatDev_ProgramDev2_GPT4o_99_scored.json",
  "trajectory\\ChatDev_ProgramDev2_GPT4o_99_scored.json"
]
Sample: 0.0\ChatDev_ProgramDev2_GPT4o_0_scored.json
Sample scoring_context:
{
  "user_demand_in_prompt": true,
  "premise_order": [
    "tasks",
    "user_demand"
  ],
  "available_premise_order": [
    "user_demand",
    "known_context",
    "tasks"
  ],
  "judge_rule": "Yes iff at least one premise supports the output and no premise contradicts it; No if any premise contradicts it or if no premise supports it."
}


In [5]:
# Average scores by classification label for selected 0.0/1.1-related outputs only.
import csv

selected_scored_paths = sorted({scored_output_path(src_path) for src_path in discover_summarized_paths()})
summary = {}
for scored_path in selected_scored_paths:
    if not scored_path.exists():
        continue
    label = scored_path.relative_to(OUTPUT_ROOT).parts[0]
    payload = json.loads(scored_path.read_text(encoding="utf-8"))
    label_stats = summary.setdefault(label, {
        "label": label,
        "files": 0,
        "entries": 0,
        "outputs": 0,
        "entry_score_sum": 0.0,
        "output_score_sum": 0.0,
    })
    label_stats["files"] += 1
    for interactions in payload.values():
        if not isinstance(interactions, list):
            continue
        for entry in interactions:
            if not isinstance(entry, dict):
                continue
            entry_mean = entry.get("consistency_score_mean")
            if isinstance(entry_mean, (int, float)):
                label_stats["entries"] += 1
                label_stats["entry_score_sum"] += float(entry_mean)
            for item in entry.get("output_consistency_scores") or []:
                score = item.get("score") if isinstance(item, dict) else None
                if isinstance(score, (int, float)):
                    label_stats["outputs"] += 1
                    label_stats["output_score_sum"] += float(score)

rows = []
for label, stats in sorted(summary.items()):
    rows.append({
        "label": label,
        "files": stats["files"],
        "entries": stats["entries"],
        "outputs": stats["outputs"],
        "entry_score_mean": stats["entry_score_sum"] / stats["entries"] if stats["entries"] else None,
        "output_score_mean": stats["output_score_sum"] / stats["outputs"] if stats["outputs"] else None,
        "premise_order": ",".join(PREMISE_ORDER),
    })

write_json(OUTPUT_ROOT / "_label_score_summary.json", rows)
with (OUTPUT_ROOT / "_label_score_summary.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["label", "files", "entries", "outputs", "entry_score_mean", "output_score_mean", "premise_order"])
    writer.writeheader()
    writer.writerows(rows)

print(json.dumps(rows, ensure_ascii=False, indent=2))


[
  {
    "label": "0.0",
    "files": 37,
    "entries": 513,
    "outputs": 4898,
    "entry_score_mean": 0.6415984163983725,
    "output_score_mean": 0.5588087475739414,
    "premise_order": "tasks,user_demand"
  },
  {
    "label": "1.1",
    "files": 32,
    "entries": 430,
    "outputs": 3984,
    "entry_score_mean": 0.6206128486930061,
    "output_score_mean": 0.5626762780004095,
    "premise_order": "tasks,user_demand"
  },
  {
    "label": "1.3",
    "files": 11,
    "entries": 150,
    "outputs": 1323,
    "entry_score_mean": 0.6154868482389899,
    "output_score_mean": 0.5790006065289612,
    "premise_order": "tasks,user_demand"
  },
  {
    "label": "1.4",
    "files": 1,
    "entries": 13,
    "outputs": 124,
    "entry_score_mean": 0.44134676399816064,
    "output_score_mean": 0.41628925894870755,
    "premise_order": "tasks,user_demand"
  },
  {
    "label": "1.5",
    "files": 12,
    "entries": 163,
    "outputs": 1443,
    "entry_score_mean": 0.6048145646025597,
    "